In [4]:
import pandas as pd
df = pd.read_csv('train.csv')
df.head()

,month_churn,nid,report_date,v_0,v_1,v_10,v_100,v_101,v_102,v_103,...,v_91,v_92,v_93,v_94,v_95,v_96,v_97,v_98,v_99,week_churn
0,0,10363,2024-09-08,45,48,344,7,5,5,0,...,0,0,45,48,0,3,7,7,6,0
1,0,3303,2024-10-27,35,42,47,45,43,45,0,...,0,1,46,275,1,1,47,47,45,1
2,0,9732,2023-10-29,35,39,19,19,6,6,0,...,0,0,45,76,1,0,19,19,19,1
3,1,2934,2024-12-08,11,15,101,91,31,48,0,...,0,1,122,460,0,0,93,93,84,1
4,0,3251,2024-03-10,3,3,181,33,23,23,0,...,0,0,29,73,1,2,33,33,32,0


In [1]:
import pandas as pd
import os
import numpy as np

# читаем исходный датасет
df = pd.read_csv("train.csv")

# считаем количество строк на каждый nid
nid_counts = df["nid"].value_counts().reset_index()
nid_counts.columns = ["nid", "count"]

# сортируем по количеству строк
nid_counts = nid_counts.sort_values("count").reset_index(drop=True)

# задаем желаемое количество частей
num_parts = 10  # можно поменять

# считаем общее количество строк и целевой размер части
total_rows = nid_counts["count"].sum()
target_rows = total_rows / num_parts

# разбиваем по логике "равных сумм строк", но сохраняя близкие count
parts = []
current_part = []
current_sum = 0

for _, row in nid_counts.iterrows():
    nid, count = row["nid"], row["count"]
    if current_sum + count > target_rows and len(parts) < num_parts - 1:
        parts.append(current_part)
        current_part = [nid]
        current_sum = count
    else:
        current_part.append(nid)
        current_sum += count

# добавляем последний кусок
if current_part:
    parts.append(current_part)

for i, part in enumerate(parts, 1):
    part_df = df[df["nid"].isin(part)]
    stats = part_df["nid"].value_counts()
    part_min = stats.min()
    part_max = stats.max()

    # считаем количество нулей
    week_zeros = (part_df["week_churn"] == 0).sum()
    month_zeros = (part_df["month_churn"] == 0).sum()

    print(f"\n=== part_{i} ===")
    print(f"Количество уникальных nid: {len(stats)}")
    print(f"Диапазон строк на nid: {part_min}–{part_max}")
    print(f"Всего строк: {len(part_df)}")
    print(f"Нулей в week_churn: {week_zeros} ({week_zeros / len(part_df) * 100:.2f}%)")
    print(f"Нулей в month_churn: {month_zeros} ({month_zeros / len(part_df) * 100:.2f}%)")



=== part_1 ===
Количество уникальных nid: 5356
Диапазон строк на nid: 1–12
Всего строк: 24810
Нулей в week_churn: 10407 (41.95%)
Нулей в month_churn: 12048 (48.56%)

=== part_2 ===
Количество уникальных nid: 1591
Диапазон строк на nid: 12–20
Всего строк: 24811
Нулей в week_churn: 15757 (63.51%)
Нулей в month_churn: 20004 (80.63%)

=== part_3 ===
Количество уникальных nid: 1081
Диапазон строк на nid: 20–26
Всего строк: 24800
Нулей в week_churn: 17658 (71.20%)
Нулей в month_churn: 22104 (89.13%)

=== part_4 ===
Количество уникальных nid: 851
Диапазон строк на nid: 26–32
Всего строк: 24804
Нулей в week_churn: 19101 (77.01%)
Нулей в month_churn: 22517 (90.78%)

=== part_5 ===
Количество уникальных nid: 684
Диапазон строк на nid: 32–41
Всего строк: 24792
Нулей в week_churn: 18720 (75.51%)
Нулей в month_churn: 22023 (88.83%)

=== part_6 ===
Количество уникальных nid: 544
Диапазон строк на nid: 41–50
Всего строк: 24792
Нулей в week_churn: 18909 (76.27%)
Нулей в month_churn: 22446 (90.54%)

=